# 01.1 — Syntax basics

Quickfire tour of Python syntax for someone who already writes MATLAB. This notebook isn't trying to teach you to program — it's translating between two notations.

After this notebook you should be able to read most of `src/neural_data_decoding/` without getting tripped up by syntax-level differences. The more interesting differences (objects, scoping, mutability) are covered in [00.3 the MATLAB → Python mental model](../00_orientation/00.3_the_matlab_to_python_mental_model.ipynb).

## Section 1 — What MATLAB does

A typical MATLAB script looks like this:

```matlab
% Comments start with %.
x = 5;                     % suppress output with ;
y = 3.14;                  % no type declarations
name = 'cgerrity';         % strings use single quotes

% Arithmetic
z = x + y;
w = x .* y;                % element-wise multiplication

% Display
fprintf('z = %.2f\n', z);
disp(w);

% If/end blocks
if z > 5
    fprintf('big\n');
else
    fprintf('small\n');
end
```

Python's equivalent uses largely the same operators but with several syntax differences. Let's walk through each.

## Section 2 — The Python concepts you need

### 2.1 — Variables, comments, output

Python uses `#` for comments, no `;` at end of lines (newlines separate statements), and `print(...)` for output.

In [ ]:
# Comment
x = 5
y = 3.14
name = "cgerrity"     # strings use double or single quotes; both work

# Arithmetic
z = x + y
w = x * y             # NOT element-wise — that's `*` on scalars; numpy has `.*` analog

# Display
print(f"z = {z:.2f}")
print(w)

**Differences worth pinning:**

| MATLAB | Python | Notes |
|---|---|---|
| `%` | `#` | comments |
| `;` (suppress) | (nothing) | `print()` is explicit |
| `'foo'` | `"foo"` or `'foo'` | strings; both quote styles work |
| `.*` | `*` (on numpy arrays) | element-wise needs numpy in Python |
| `fprintf('z = %.2f\n', z)` | `print(f"z = {z:.2f}")` | f-strings |
| `disp(w)` | `print(w)` | same idea |
| `end` | indentation | no closing keyword |


### 2.2 — Numbers, strings, booleans

In [ ]:
# Numbers (int and float are distinct types)
a = 5         # int
b = 5.0       # float
c = 1 / 2     # 0.5 — division always returns float
d = 1 // 2    # 0 — integer division (floor)
e = 7 % 3     # 1 — modulo
f = 2 ** 10   # 1024 — exponentiation (NOT ^, which is XOR!)

print(a, b, c, d, e, f)

In [ ]:
# Strings
s = "hello"
n = len(s)            # length — built-in function
print(s.upper())      # methods via the dot
print(s[0])           # first character (0-indexed)
print(s[-1])          # last character
print("hello" + " " + "world")    # concatenation with +
print(f"{n} characters in {s!r}")  # f-strings — !r calls repr()

In [ ]:
# Booleans
t = True              # capital T/F
f = False
print(t and f)        # logical and — not &&
print(t or f)         # logical or — not ||
print(not t)          # negation — not !
print(t == 1, f == 0) # True == 1 and False == 0 are both True in Python

**The `^` trap.** In MATLAB `^` is exponentiation. In Python `^` is **bitwise XOR**. Use `**` for exponentiation:

* `2 ^ 10` in Python returns `8`. (Surprising.)
* `2 ** 10` in Python returns `1024`. (What you want.)

This bites every MATLAB user at least once. The error is silent — no exception, just wrong numbers — so audit any port that uses `^`.

### 2.3 — Collections: lists, tuples, dicts, sets

Python has four core built-in collection types. MATLAB's cell arrays and structs cover similar ground but the semantics differ.

In [ ]:
# Lists — ordered, mutable, heterogeneous. MATLAB analog: cell array {}.
fruits = ["apple", "banana", "cherry"]
fruits.append("date")
fruits[0] = "apricot"
print(fruits)
print(len(fruits), fruits[-1])

In [ ]:
# Tuples — like lists but IMMUTABLE. Cheap, hashable. Pythonic for fixed-shape records.
point = (3, 4)
x, y = point          # tuple unpacking
print(f"x={x}, y={y}")

# You CAN'T do point[0] = 99 — tuples are immutable.
try:
    point[0] = 99
except TypeError as e:
    print(f"Error: {e}")

In [ ]:
# Dicts — key→value mappings. MATLAB analog: struct (with string keys), or containers.Map.
person = {"name": "Charles", "role": "PI", "years": 12}
print(person["name"])         # access by key
person["lab"] = "VU"          # add a new key
del person["years"]           # delete a key

for key, value in person.items():
    print(f"  {key} = {value}")

In [ ]:
# Sets — unordered, no duplicates. MATLAB analog: unique(...) result.
seen_models = {"GRU", "LSTM", "GRU", "Feedforward"}
print(seen_models)              # {'GRU', 'LSTM', 'Feedforward'} — duplicates removed
seen_models.add("Resnet")
print("GRU" in seen_models)     # O(1) membership test

**When to use which:**

| Need | Pick | Why |
|---|---|---|
| Ordered + mutable | `list` | the default Python collection |
| Fixed-shape record, hashable | `tuple` | `(x, y)`, `(min, max)` pairs |
| Lookup by name | `dict` | MATLAB struct equivalent |
| Membership testing only | `set` | O(1) `in` checks |

In our codebase: cfg fields are dicts (`cfg.dropout` via `omegaconf`), hidden sizes are lists (`[1000, 500, 250]`), shape descriptors are tuples (`(W, T, A, C)`), supported model names are dict keys (registry pattern).

### 2.4 — String formatting

f-strings are the modern way to interpolate values into strings (Python 3.6+). They replace MATLAB's `sprintf`/`fprintf` family.

In [ ]:
width = 100
stride = 50
lr = 1e-3

# Plain interpolation
print(f"Data Width - {width}")

# Field width and precision
print(f"LR = {lr:.2e}")          # 1.00e-03 — MATLAB's %.2e
print(f"width={width:>5d}")       # right-aligned in 5 chars

# Inline expressions
print(f"Total cells: {width * stride}")

# !r for repr (shows quotes around strings)
config = "Optimal"
print(f"using {config!r}")

## Section 3 — The `neural_data_decoding` implementation

Look at a real production file to see most of these syntactic constructs in one place. The sweep dispatcher's per-entry data layer is mostly literals — strings, numbers, lists, dicts, tuples — which makes it a good first read.

In [ ]:
from pathlib import Path
# Show the first 40 lines of the dispatcher module
src = Path("../../src/neural_data_decoding/sweeps/dispatcher.py").read_text().splitlines()
for i, line in enumerate(src[:40], start=1):
    print(f"{i:3} | {line}")

Things to spot:
- Triple-quoted module docstring at the top — Python's equivalent of a MATLAB `%CGG_*` header comment.
- `from __future__ import annotations` — enables modern type-hint syntax. Always include in new modules.
- `from collections.abc import Iterator` — importing types from the stdlib.
- `dict[str, Any]` — type hint for a dict from string keys to anything. (Python 3.9+ syntax.)
- The `_MATLAB_TO_PYTHON_FIELD` dict literal — exactly the kind of static mapping you'd build with a MATLAB struct.

## Section 4 — Hands-on exercises

### Exercise 4.1 — translate MATLAB to Python

Port this snippet:

```matlab
x = 5;
y = 10;
if x + y > 12
    fprintf('sum = %d\n', x + y);
else
    fprintf('small\n');
end
```

Try it in the cell below before unhiding the answer.

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
x = 5
y = 10
if x + y > 12:
    print(f"sum = {x + y}")
else:
    print("small")

### Exercise 4.2 — build a config dict

Build a Python dict that has the same fields as a MATLAB cfg struct for a GRU encoder: `model_name='GRU'`, `hidden_sizes=[1000, 500, 250]`, `dropout=0.5`, `is_variational=True`.

Then print just the hidden-sizes list.

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
cfg = {
    "model_name": "GRU",
    "hidden_sizes": [1000, 500, 250],
    "dropout": 0.5,
    "is_variational": True,
}
print(cfg["hidden_sizes"])

## Section 5 — Common errors

### `SyntaxError: invalid syntax`

Most often a missing colon at the end of an `if`/`for`/`while`/`def`/`class` line, or extra parentheses. Python is strict about colons:

```python
if x > 0:           # ← colon required
    print("hi")
```

### `IndentationError: expected an indented block`

You wrote `if x > 0:` but didn't indent the body. Or you indented inconsistently (4 spaces in one line, 2 in the next).

### `^` does the wrong thing

Use `**` for exponentiation. See Section 2.2.

### `KeyError: 'foo'`

You tried `cfg["foo"]` for a key that doesn't exist. Use `cfg.get("foo", default)` if missing keys are OK, or `if "foo" in cfg: ...` to test first.

### `TypeError: 'tuple' object does not support item assignment`

Tuples are immutable. Use a list `[1, 2, 3]` if you need to mutate.

### `NameError: name 'fprintf' is not defined`

`fprintf` is a MATLAB built-in. Python's equivalent is `print(f"..."")`. The `f"..."` is an f-string.

## Section 6 — Further reading

- [Python tutorial — informal intro](https://docs.python.org/3/tutorial/index.html). The official tutorial; readable in an afternoon.
- [PEP 8 — Style Guide](https://peps.python.org/pep-0008/). The 4-space-indent, snake_case-naming conventions everyone uses.
- [PEP 498 — f-strings](https://peps.python.org/pep-0498/). The modern formatting syntax.
- [Real Python — common gotchas](https://realpython.com/python-gotchas/). Includes the `^` vs `**` trap and the mutable-default-argument bug.

Next up: **[01.2 control flow](01.2_control_flow.ipynb)**.